# Practical 03 — Land Use / Land Cover Dynamics with AAFC Annual Crop Inventory

**Course:** Satellite Remote Sensing Practicals  
**Level:** Advanced undergraduate / graduate  
**Platforms:** Google Colab · Google Earth Engine (students' own projects) · Google Drive (students' own accounts)  
**Study region:** A student-drawn agricultural landscape in Canada (Manitoba recommended)  
**Requested inventory interval:** 2010–2025, automatically restricted to years actually available in Earth Engine.

## Scientific question
**How do crop types and other land-cover categories change from year to year, and what evidence is needed to distinguish annual crop switching from potentially persistent landscape conversion?**

### Learning outcomes
1. Access and interpret the AAFC Annual Crop Inventory (ACI) and its land-cover codes.
2. Define a study area interactively using an imagery basemap.
3. Quantify annual class areas in hectares and percentages.
4. Examine interannual crop composition and year-to-year crop class switching.
5. Construct transition matrices and Sankey diagrams for selected years.
6. Map candidate grassland-to-cropland and cropland-to-grassland conversion using multi-year persistence rules.
7. Discuss classification uncertainty, pixel size, temporal gaps, and methodological limitations.

**Scientific caution:** AAFC ACI is an annual *classification product*, not ground-truth crop management or observed land conversion. A difference in crop label between two dates is not proof of a permanent land-cover conversion. The 2010 inventory has a different native resolution (~56 m) from 2011 onward (~30 m). Maps and area summaries use a specified nominal 30 m processing scale; do **not** interpret 2010's resampled pixels as genuine 30 m observations.

**Verified data documentation:** https://developers.google.com/earth-engine/datasets/catalog/AAFC_ACI  
**AAFC 2025 inventory (official external publication):** https://open.canada.ca/data/en/dataset/89073037-d2fc-4a6d-9ce9-f0b1bcb48745  
**EE dataset ID:** `AAFC/ACI` · band `landcover` · data license: Open Government Licence – Canada 2.0.  

**Processing costs:** Use a manageable ROI initially (recommended 10 × 10 to 20 × 20 km). All per-year/pair summaries are requested sequentially to avoid the concurrent-aggregation error from Practical 2.

## 0. Package check — do not upgrade Colab dependencies

**Do not run** `pip install -U earthengine-api geemap pandas ...`. A standard fresh Colab runtime already includes the principal scientific libraries. Upgrading packages can create incompatible `google-auth` and `pandas` requirements. This notebook checks imports without changing the environment.

In [ ]:
from importlib import util
REQUIRED = ['ee', 'geemap', 'numpy', 'pandas', 'matplotlib', 'PIL', 'shapely']
missing = [name for name in REQUIRED if util.find_spec(name) is None]
if missing:
    raise RuntimeError('Missing Colab libraries: ' + ', '.join(missing) +
                       '. Open a fresh standard Google Colab runtime; do not bulk-upgrade dependencies.')
print('Required libraries are available. No packages were installed or upgraded.')

In [ ]:
import os, re, json, hashlib, time, warnings, urllib.request
from pathlib import Path
from io import BytesIO
from datetime import datetime, timezone
from collections import OrderedDict

import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from PIL import Image
from IPython.display import display
from shapely.geometry import shape, mapping

try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    print('Plotly unavailable; Sankey diagrams will be skipped, CSV matrices still work.')

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except (ImportError, AttributeError):
    pass

plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 300,
                     'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.15,
                     'axes.spines.top': True, 'axes.spines.right': True})
print('Imports successful.')

## 1. Student configuration — edit your account and study settings

Each student must use their **own** Earth Engine-registered Google Cloud project, and should run Colab with their own Google account. Every CSV/PNG/HTML/GeoJSON file is written into their mounted **My Drive**. Earth Engine GeoTIFF tasks use a separate top-level Drive folder; monitor tasks until they finish.

In [ ]:
#@title Practical 3 — Student configuration
# ===== REQUIRED STUDENT ENTRIES =====
EE_PROJECT_ID = 'CHANGE_ME'  #@param {type:"string"} # Your Earth Engine-enabled project
STUDENT_ID = 'FirstName_LastName'  #@param {type:"string"} # Your own Drive folder

# ===== ANALYSIS SETTINGS =====
START_YEAR = 2010
END_YEAR = 2025                      # Automatically skip unavailable EE years
PROCESSING_SCALE_M = 30              # 2010 native resolution is ~56 m (important caveat)
MAX_ROI_KM2 = 2500                    # Classroom safeguard (change only with instructor approval)

# Prefer 2011+ for transition/change mapping to avoid 2010 resolution mismatch.
REQUESTED_CHANGE_FROM = 2011
REQUESTED_CHANGE_TO = 2024
PERSISTENCE_WINDOW = 3               # Consecutive early/late years for robust candidates
TOP_N_DETAILED_CLASSES = 10
SANKey_MIN_FLOW_HA = 1.0

# Export controls (be mindful of Earth Engine task limits)
MAKE_STATIC_MAPS = True              # EE thumbnail attempts, optional if blocked
START_GEOTIFF_EXPORTS = True         # EE tasks are asynchronous
EXPORT_ALL_ANNUAL_GEOTIFFS = False   # Set True only for a small ROI and if required

if not EE_PROJECT_ID.strip() or EE_PROJECT_ID == 'CHANGE_ME':
    raise ValueError('Enter your OWN EE_PROJECT_ID in this cell.')
if not STUDENT_ID.strip() or STUDENT_ID == 'FirstName_LastName':
    raise ValueError('Enter your own STUDENT_ID.')
if START_YEAR >= END_YEAR or START_YEAR < 2009:
    raise ValueError('Choose a valid multiyear range within the ACI record.')
if PROCESSING_SCALE_M <= 0:
    raise ValueError('PROCESSING_SCALE_M must be positive.')

TAG = re.sub(r'[^A-Za-z0-9_-]+', '_', STUDENT_ID).strip('_')[:55]
try:
    ee.Initialize(project=EE_PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT_ID)
print(f'Earth Engine project: {EE_PROJECT_ID} | Student: {TAG}')

## 2. Mount the student's own Google Drive

When prompted, authorize **your own** Google Drive. A separate subfolder is created for this ROI; this protects past results if you change the study area later.

In [ ]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError('This notebook is designed for Google Colab.') from exc

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/Satellite_Practicals/Practical_03') / TAG
BASE.mkdir(parents=True, exist_ok=True)
print('Student output root:', BASE)
print('A study-specific subfolder will be created after ROI selection.')

## 3. Draw one ROI interactively (or use GeoJSON/BBOX as a fallback)

**Recommended:** Run the next cell, zoom to an agricultural area and draw **one polygon or rectangle** on the satellite imagery. Finish the drawing, then run the following capture cell. Do **not** re-run the map-creation cell after drawing; this can reset the map widget.

The default background uses Esri World Imagery to avoid the OpenStreetMap tile access error experienced in earlier practicals. If the map widget is blocked, switch to `geojson` or `bbox` input. The example center only positions the map; students select their own ROI.

In [ ]:
ROI_INPUT_MODE = 'draw'          # 'draw', 'geojson', or 'bbox'
GEOJSON_PATH = '/content/study_area.geojson'
BBOX_W_S_E_N = None             # For bbox mode: [west, south, east, north] (degrees)

roi_map = None
if ROI_INPUT_MODE == 'draw':
    try:
        roi_map = geemap.Map(center=[49.60, -98.18], zoom=11,
                             basemap='Esri.WorldImagery', draw_ctrl=True)
        display(roi_map)
        print('Draw ONE polygon/rectangle, finish the drawing, then run the capture cell.')
        print('If you draw multiple polygons they will be combined into one study area.')
    except Exception as exc:
        print('Map widget failed:', exc)
        print("Use ROI_INPUT_MODE='geojson' or 'bbox'; then rerun this cell.")

In [ ]:
def polygon_geometries(data):
    """Recursively unpack FeatureCollection/Feature/GeometryCollection and polygonal geometry."""
    if not isinstance(data, dict):
        return []
    kind = data.get('type')
    if kind == 'FeatureCollection':
        out = []
        for feat in data.get('features', []):
            out += polygon_geometries(feat)
        return out
    if kind == 'Feature':
        return polygon_geometries(data.get('geometry', {}))
    if kind == 'GeometryCollection':
        out = []
        for geom in data.get('geometries', []):
            out += polygon_geometries(geom)
        return out
    return [data] if kind in ['Polygon', 'MultiPolygon'] else []


def drawn_roi_geometries(m):
    """Read all drawings from the current geemap widget (version-compatible)."""
    candidate_lists = []
    rois = getattr(m, 'user_rois', None)
    if rois is not None:
        try:
            candidate_lists.append(polygon_geometries(ee.FeatureCollection(rois).getInfo()))
        except Exception as exc:
            print('Could not read user_rois:', exc)
    ctrl = getattr(m, 'draw_control', None) or getattr(m, '_draw_control', None)
    collection = getattr(ctrl, 'collection', None) if ctrl is not None else None
    if collection is not None:
        try:
            candidate_lists.append(polygon_geometries(ee.FeatureCollection(collection).getInfo()))
        except Exception as exc:
            print('Could not read draw-control collection:', exc)
    drawn = getattr(m, 'draw_features', None)
    if drawn is not None:
        geometries = []
        for feature in drawn:
            if isinstance(feature, dict):
                geometries += polygon_geometries(feature)
            else:
                try:
                    geometries += polygon_geometries(ee.Feature(feature).geometry().getInfo())
                except Exception:
                    pass
        candidate_lists.append(geometries)
    return max(candidate_lists, key=len) if candidate_lists else []

if ROI_INPUT_MODE == 'draw':
    if roi_map is None:
        raise RuntimeError('Map was not created. Change ROI_INPUT_MODE to geojson or bbox.')
    geometries = drawn_roi_geometries(roi_map)
elif ROI_INPUT_MODE == 'geojson':
    with open(GEOJSON_PATH, encoding='utf-8') as f:
        geometries = polygon_geometries(json.load(f))
elif ROI_INPUT_MODE == 'bbox':
    if BBOX_W_S_E_N is None or len(BBOX_W_S_E_N) != 4:
        raise ValueError('Set BBOX_W_S_E_N = [west, south, east, north].')
    w,s,e,n = map(float, BBOX_W_S_E_N)
    if not (w < e and s < n):
        raise ValueError('Bounding box needs west < east and south < north.')
    geometries = [{'type': 'Polygon', 'coordinates': [[[w,s],[e,s],[e,n],[w,n],[w,s]]]}]
else:
    raise ValueError('ROI_INPUT_MODE must be draw, geojson, or bbox.')

if not geometries:
    raise ValueError('No ROI polygon found. Finish drawing on the EXISTING map and rerun ONLY this capture cell.')

polygons = [shape(g) for g in geometries]
if any(p.is_empty or not p.is_valid or p.geom_type not in ('Polygon', 'MultiPolygon') for p in polygons):
    raise ValueError('One or more ROI geometries are invalid. Redraw simple non-self-intersecting polygons.')

# Union separate drawn polygons using a FeatureCollection; preserves disjoint shapes.
# This also supports holes and MultiPolygon inputs without dropping coordinates.
ROI_EE = ee.FeatureCollection(
    [ee.Feature(ee.Geometry(g)) for g in geometries]).geometry(maxError=1)

ROI_AREA_KM2 = float(ROI_EE.area(maxError=1).divide(1e6).getInfo())
if ROI_AREA_KM2 <= 0 or ROI_AREA_KM2 > MAX_ROI_KM2:
    raise ValueError(f'ROI = {ROI_AREA_KM2:.1f} km²; select a positive ROI <= {MAX_ROI_KM2} km².')

roi_geojson = ROI_EE.getInfo()
roi_fingerprint = hashlib.sha1(json.dumps(roi_geojson, sort_keys=True).encode('utf-8')).hexdigest()[:10]
RUN_ROOT = BASE / f'ROI_{roi_fingerprint}'
DIRS = {name: RUN_ROOT / name for name in ['ROI','Tables','Figures','Maps','GeoTIFF_Logs']}
for folder in DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)
with open(DIRS['ROI'] / 'study_area.geojson', 'w', encoding='utf-8') as f:
    json.dump({'type':'Feature','properties':{'area_km2':ROI_AREA_KM2},'geometry':roi_geojson},f,indent=2)
print(f'ROI captured: {len(geometries)} drawn geometry(ies), {ROI_AREA_KM2:.2f} km²')
print('All immediate outputs will be saved to:', RUN_ROOT)

## 4. Retrieve AAFC inventories and verify available years

The official 2025 ACI publication and the **Earth Engine collection** are distinct distribution channels. This code queries EE collection metadata and uses only years actually present; it will **explicitly list missing years**. 2010 is retained for annual-area summaries, but the default conversion pair starts in 2011 to avoid comparing 56 m and 30 m native products.

In [ ]:
ACI = ee.ImageCollection('AAFC/ACI')
times_ms = ACI.aggregate_array('system:time_start').getInfo()
CATALOG_YEARS = sorted({datetime.fromtimestamp(int(t)/1000, tz=timezone.utc).year
                        for t in times_ms if t is not None})
REQUESTED_YEARS = list(range(START_YEAR, END_YEAR + 1))
YEARS = [y for y in REQUESTED_YEARS if y in CATALOG_YEARS]
MISSING_YEARS = [y for y in REQUESTED_YEARS if y not in CATALOG_YEARS]
if len(YEARS) < 2:
    raise RuntimeError(f'Insufficient ACI EE years for your request. Available: {CATALOG_YEARS}')
print('Available in GEE:', YEARS)
if MISSING_YEARS:
    print('NOT available in this GEE collection (skipping):', MISSING_YEARS)
    print('An externally published ACI year is not assumed to be ingested in Earth Engine.')
if 2010 in YEARS:
    print('CAUTION: 2010 has ~56 m native spatial resolution; 2011 onward ~30 m.')

# Date-filter one inventory image per year (not a yearly average).
ACI_IMAGES = {y: ee.Image(ACI.filterDate(f'{y}-01-01',f'{y+1}-01-01').first())
              .select('landcover') for y in YEARS}

CHANGE_FROM = REQUESTED_CHANGE_FROM if REQUESTED_CHANGE_FROM in YEARS and REQUESTED_CHANGE_FROM >= 2011 else min(y for y in YEARS if y >= 2011)
CHANGE_TO = REQUESTED_CHANGE_TO if REQUESTED_CHANGE_TO in YEARS and REQUESTED_CHANGE_TO > CHANGE_FROM else max(YEARS)
if CHANGE_TO <= CHANGE_FROM:
    raise RuntimeError('Choose at least two years after 2010 for change analysis.')
CONSECUTIVE_PAIRS = [(a, b) for a,b in zip(YEARS[:-1], YEARS[1:])
                     if b == a+1 and a >= 2011]
print(f'Long-term comparison: {CHANGE_FROM} → {CHANGE_TO}')
print(f'Consecutive crop-switching comparisons: {len(CONSECUTIVE_PAIRS)}')

## 5. Land-cover class codes, broad thematic groups and legends

Do not average category codes. Each ACI pixel has a **nominal class label**. We retain all original class values in the statistics and add a documented *teaching-level* grouping for broad landscape processes. Grouped categories are an interpretation tool, not an official replacement for the AAFC classes.

The crosswalk below follows the AAFC codebook, including spring wheat (`146`), canola (`153`), soybeans (`158`), potatoes (`177`), grassland (`110`) and pasture/forages (`122`). Cloud code `10` is treated as unclassified and excluded from area summaries and paired analyses.

In [ ]:
# Official class-value, name, and colour lookup (see linked AAFC codebook).
CLASS_INFO = {
 10:('Cloud','#000000'),20:('Water','#3333ff'),30:('Exposed Land and Barren','#996666'),
 34:('Urban and Developed','#cc6699'),35:('Greenhouses','#e1e1e1'),
 50:('Shrubland','#ffff00'),60:('Forest Fire and Burnt Area','#666666'),
 80:('Wetland','#993399'),85:('Peatland','#501b50'),110:('Grassland','#cccc00'),
 120:('Agriculture (undifferentiated)','#cc6600'),121:('Cropland','#ff9933'),
 122:('Pasture and Forages','#ffcc33'),130:('Too Wet to be Seeded','#7899f6'),
 131:('Fallow','#ff9900'),132:('Cereals','#660000'),133:('Barley','#dae31d'),
 134:('Other Grains','#99cc00'),135:('Millet','#d2db25'),136:('Oats','#d1d52b'),
 137:('Rye','#cacd32'),138:('Spelt','#c3c63a'),139:('Triticale','#b9bc44'),
 140:('Wheat','#a7b34d'),141:('Switchgrass','#b9c64e'),142:('Sorghum','#999900'),
 143:('Quinoa','#e9e2b1'),145:('Winter Wheat','#809769'),146:('Spring Wheat','#92a55b'),
 147:('Corn for Grain','#ffff99'),148:('Tobacco','#98887c'),149:('Ginseng','#799b93'),
 150:('Oilseeds','#5ea263'),151:('Borage','#52ae77'),152:('Camelina','#41bf7a'),
 153:('Canola and Rapeseed','#d6ff70'),154:('Flaxseed','#8c8cff'),
 155:('Mustard','#d6cc00'),156:('Safflower','#ff7f00'),157:('Sunflower','#315491'),
 158:('Soybeans','#cc9933'),159:('Other Oilseeds','#5ea296'),
 160:('Pulses','#896e43'),161:('Other Pulses','#996633'),162:('Peas','#8f6c3d'),
 163:('Chickpeas','#b6a472'),167:('Beans','#82654a'),168:('Fababeans','#a39069'),
 174:('Lentils','#b85900'),175:('Vegetables','#b74b15'),176:('Tomatoes','#ff8a8a'),
 177:('Potatoes','#ffcccc'),178:('Sugarbeets','#6f55ca'),179:('Other Vegetables','#ffccff'),
 180:('Fruits','#dc5424'),181:('Berries','#d05a30'),182:('Blueberry','#d20000'),
 183:('Cranberry','#cc0000'),185:('Other Berries','#dc3200'),188:('Orchards','#ff6666'),
 189:('Other Fruits','#c5453b'),190:('Vineyards','#7442bd'),191:('Hops','#ffcc99'),
 192:('Sod','#b5fb05'),193:('Herbs','#ccff05'),194:('Nursery','#07f98c'),
 195:('Buckwheat','#00ffcc'),196:('Canaryseed','#cc33cc'),197:('Hemp','#8e7672'),
 198:('Vetch','#b1954f'),199:('Other Crops','#749a66'),
 200:('Forest (undifferentiated)','#009900'),210:('Coniferous','#006600'),
 220:('Broadleaf','#00cc00'),230:('Mixedwood','#cc9900')
}

# Ordered: (group_id, name, display colour, contributing AAFC class codes)
GROUP_DEF = [
 (1,'Water','#3f77da',[20]),
 (2,'Urban / bare','#b6a2a2',[30,34,35]),
 (3,'Shrubland','#c7b841',[50]),
 (4,'Burned land','#6c6c6c',[60]),
 (5,'Wetland / peatland','#94518f',[80,85]),
 (6,'Grassland','#8db33c',[110]),
 (7,'Pasture / forage','#87ad80',[122,141,192,198]),
 (8,'Cereals / grains','#dbc443',[132,133,134,135,136,137,138,139,140,142,143,145,146,147,195,196]),
 (9,'Oilseeds','#ef9750',[150,151,152,153,154,155,156,157,158,159]),
 (10,'Pulses / legumes','#9c7050',[160,161,162,163,167,168,174]),
 (11,'Vegetables / potatoes','#d67793',[175,176,177,178,179]),
 (12,'Fruit / horticulture','#d24e75',[148,149,180,181,182,183,185,188,189,190,191,193,194,197]),
 (13,'Other agriculture','#c2a677',[120,121,130,131,199]),
 (14,'Forest','#257b4d',[200,210,220,230]),
 (15,'Other classified','#9b9b9b',[])
]

GROUP_IDS = [x[0] for x in GROUP_DEF]
GROUP_NAMES = {x[0]:x[1] for x in GROUP_DEF}
GROUP_COLORS = {x[0]:x[2] for x in GROUP_DEF}
CODE_TO_GROUP = {code:gid for gid,_,_,codes in GROUP_DEF for code in codes}
# Crop-TYPE switching must exclude generic agricultural/fallow/no-seeding codes.
CROP_GROUP_IDS = [8,9,10,11,12]
CROP_CODES = sorted([code for code,gid in CODE_TO_GROUP.items() if gid in CROP_GROUP_IDS] + [199])
# Land-use cropland persistence also includes generic cropland and fallow codes.
CROPLAND_LANDUSE_CODES = sorted(set(CROP_CODES + [121,131]))
GRASS_CODE = 110
CODES_FOR_EE = sorted(CLASS_INFO)
GROUP_IDS_FOR_EE = [0 if code == 10 else CODE_TO_GROUP.get(code,15) for code in CODES_FOR_EE]
BROAD_PALETTE = [GROUP_COLORS[i] for i in GROUP_IDS]

missing_crosswalk = sorted(set(CLASS_INFO) - set(CODE_TO_GROUP) - {10})
if missing_crosswalk:
    print('Codes without an explicit group (handled as Other classified):', missing_crosswalk)
print('Grouping contains',len(GROUP_DEF),'land-cover groups and',len(CLASS_INFO),'codebook classes.')


def raw_aci(year):
    # Preserve nominal categories and discard unclassified cloud label.
    im = ee.Image(ACI_IMAGES[year]).toInt16()
    return im.updateMask(im.neq(10))


def broad_aci(year):
    raw = raw_aci(year)
    group = raw.remap(CODES_FOR_EE, GROUP_IDS_FOR_EE, 15).rename('broad_group').toInt16()
    return group.updateMask(group.gt(0))


def area_by_class(image, geometry=ROI_EE, scale=PROCESSING_SCALE_M):
    """One server-side grouped-area aggregation per year or paired map."""
    category = ee.Image(image).toInt32().rename('class_code')
    areas = (ee.Image.pixelArea().divide(10000).rename('area_ha')
             .addBands(category).updateMask(category.mask()))
    result = areas.reduceRegion(reducer=ee.Reducer.sum().group(groupField=1,groupName='class_code'),
                                geometry=geometry, scale=scale, crs=category.projection(), maxPixels=1e9,
                                tileScale=4).get('groups').getInfo()
    return [{'class_code':int(x['class_code']), 'area_ha':float(x['sum'])}
            for x in (result or [])]

# Public Earth Engine documentation: ee.Reducer.sum().group and ee.Image.reduceRegion.

## 6. Annual crop-class area statistics, 2010–latest available year

Compute hectares for **each detailed AAFC class** once per year, sequentially, then aggregate class codes to broad groups in Python. We save a checkpoint after every completed year. If an Earth Engine quota error occurs later, rerunning the cell uses already completed years in the same ROI folder. Each year's percentages use the **valid classified area**, which may be slightly smaller than the ROI polygon area.

In [ ]:
AREA_CHECKPOINT = DIRS['Tables']/'annual_detailed_area_checkpoint.csv'
AREA_COMPLETED = DIRS['Tables']/'annual_years_completed.json'

area_rows = pd.read_csv(AREA_CHECKPOINT).to_dict('records') if AREA_CHECKPOINT.exists() else []
completed = set(json.loads(AREA_COMPLETED.read_text())) if AREA_COMPLETED.exists() else set()
for year in YEARS:
    if year in completed:
        print(f'{year}: checkpoint already saved; skipping.')
        continue
    # EE request runs to completion before we launch the next year.
    result = area_by_class(raw_aci(year))
    area_rows.extend({'year':year, **r} for r in result)
    pd.DataFrame(area_rows, columns=['year','class_code','area_ha']).to_csv(AREA_CHECKPOINT,index=False)
    completed.add(year)
    AREA_COMPLETED.write_text(json.dumps(sorted(completed),indent=2))
    print(f'{year}: {len(result)} classes; saved checkpoint.')

raw_area_df = pd.DataFrame(area_rows)
if raw_area_df.empty:
    raise RuntimeError('No classified pixels in the ROI. Choose an AOI inside Canadian ACI coverage.')
raw_area_df = raw_area_df[raw_area_df.year.isin(YEARS)].copy()
raw_area_df['class_code'] = raw_area_df.class_code.astype(int)
raw_area_df['class_name'] = raw_area_df.class_code.map(lambda c: CLASS_INFO.get(c,(f'Unknown class {c}',))[0])
raw_area_df['group_id'] = raw_area_df.class_code.map(lambda c: CODE_TO_GROUP.get(c,15))
raw_area_df['group_name'] = raw_area_df.group_id.map(GROUP_NAMES)
raw_area_df['year_valid_area_ha'] = raw_area_df.groupby('year').area_ha.transform('sum')
raw_area_df['pct_of_valid_area'] = 100*raw_area_df.area_ha/raw_area_df.year_valid_area_ha
raw_area_df.to_csv(DIRS['Tables']/'annual_detailed_class_areas.csv',index=False)

group_area_df = (raw_area_df.groupby(['year','group_id','group_name'],as_index=False)
                 .agg(area_ha=('area_ha','sum')))
yr_total = group_area_df.groupby('year').area_ha.transform('sum')
group_area_df['pct_of_valid_area'] = 100*group_area_df.area_ha/yr_total
group_area_df.to_csv(DIRS['Tables']/'annual_broad_group_areas.csv',index=False)

coverage = (raw_area_df.groupby('year',as_index=False).area_ha.sum()
            .rename(columns={'area_ha':'classified_ha'}))
coverage['roi_polygon_ha'] = ROI_AREA_KM2*100
coverage['classified_pct_of_roi'] = 100*coverage.classified_ha/coverage.roi_polygon_ha
coverage.to_csv(DIRS['Tables']/'classified_area_coverage_by_year.csv',index=False)

unknowns = raw_area_df[~raw_area_df.class_code.isin(CLASS_INFO)]
if not unknowns.empty:
    print('WARNING: Unknown class codes in the data. Review crosswalk:', sorted(unknowns.class_code.unique()))
print('Area summaries saved for',group_area_df.year.nunique(),'years.')
display(coverage.round(2))
display(group_area_df[group_area_df.year==max(YEARS)].sort_values('area_ha',ascending=False).head(12).round(2))

## 7. Annual trends: crop types and broad land cover

The first plot is a stacked **percentage of classified area** by broad class. The second displays areas for the most prevalent detailed crop/land-cover codes. We label units and use consistent category colours across years.

In [ ]:
def save_figure(filename):
    plt.savefig(DIRS['Figures']/filename, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

broad = group_area_df.pivot_table(index='year',columns='group_id',values='pct_of_valid_area',fill_value=0)
broad = broad.reindex(index=YEARS,columns=GROUP_IDS,fill_value=0)
fig, ax = plt.subplots(figsize=(13,6.2))
ax.stackplot(broad.index, *[broad[i].values for i in GROUP_IDS],
             labels=[GROUP_NAMES[i] for i in GROUP_IDS],
             colors=BROAD_PALETTE, linewidth=0)
ax.set(title='AAFC ACI: annual land-cover composition',xlabel='Inventory year',
       ylabel='Share of classified ROI area (%)',ylim=(0,100))
ax.set_xticks(YEARS[::max(1,len(YEARS)//10)])
ax.legend(loc='center left',bbox_to_anchor=(1.01,0.5),fontsize=8,ncol=1)
save_figure('01_broad_landcover_stacked_percentages.png')

# Top detailed codes by total area across years; compare hectares, not arbitrary class labels.
top_codes = (raw_area_df.groupby('class_code').area_ha.sum()
             .nlargest(TOP_N_DETAILED_CLASSES).index.tolist())
top_area = (raw_area_df[raw_area_df.class_code.isin(top_codes)]
            .pivot_table(index='year',columns='class_code',values='area_ha',fill_value=0)
            .reindex(index=YEARS,columns=top_codes,fill_value=0))
fig,ax=plt.subplots(figsize=(12,5.5))
for code_val in top_codes:
    ax.plot(top_area.index, top_area[code_val],marker='o',markersize=3,
            label=CLASS_INFO.get(code_val,(str(code_val),))[0] + f' ({code_val})')
ax.set(title='Annual area: most prevalent detailed ACI classes',
       xlabel='Inventory year',ylabel='Mapped area (ha)')
ax.legend(loc='center left',bbox_to_anchor=(1.01,0.5),fontsize=8)
save_figure('02_top_detailed_classes_ha.png')
print('Saved figures to',DIRS['Figures'])

## 8. Year-to-year crop-class switching (2011 onward)

**What this measures:** For each consecutive year pair, consider only pixels assigned to *specific crop classes in both years* (excluding generic cropland, fallow, too-wet-to-seed labels). Report (a) area with the **same detailed ACI crop code** in both years and (b) area assigned to **different crop codes**. These are **mapped class changes**, not verified rotations or crop management histories.

Fallow and generically classified agricultural pixels are excluded from crop-*type* switching, but a fallow field may still count as existing cropland in the separate persistent-land-use analysis. We deliberately exclude 2010 from this consecutive-pair diagnostic because its native pixel size differs. Every pair is processed in a **separate Earth Engine request**, with checkpointing.

In [ ]:
PAIR_CHECKPOINT = DIRS['Tables']/'consecutive_crop_switching_checkpoint.csv'
PAIR_COMPLETED = DIRS['Tables']/'crop_pairs_completed.json'
crop_rows = pd.read_csv(PAIR_CHECKPOINT).to_dict('records') if PAIR_CHECKPOINT.exists() else []
completed_pairs = {tuple(x) for x in json.loads(PAIR_COMPLETED.read_text())} if PAIR_COMPLETED.exists() else set()

def crop_flag(year):
    return raw_aci(year).remap(CROP_CODES, [1]*len(CROP_CODES), 0).eq(1)

def cropland_landuse_flag(year):
    return raw_aci(year).remap(CROPLAND_LANDUSE_CODES, [1]*len(CROPLAND_LANDUSE_CODES), 0).eq(1)


def crop_pair_status(a,b):
    same_eligible = crop_flag(a).And(crop_flag(b))
    status = ee.Image.constant(1).where(raw_aci(a).neq(raw_aci(b)),2)
    return status.updateMask(same_eligible).rename('crop_switch_status').toInt16()

for a,b in CONSECUTIVE_PAIRS:
    if (a,b) in completed_pairs:
        print(f'{a}–{b}: checkpoint already saved.')
        continue
    area_records = area_by_class(crop_pair_status(a,b))
    area_dict = {r['class_code']:r['area_ha'] for r in area_records}
    same_ha = area_dict.get(1,0.)
    switched_ha = area_dict.get(2,0.)
    total_ha = same_ha+switched_ha
    crop_rows.append({'from_year':a,'to_year':b,'same_crop_code_ha':same_ha,
                      'different_crop_code_ha':switched_ha,
                      'comparable_cropland_ha':total_ha,
                      'different_pct':100*switched_ha/total_ha if total_ha>0 else np.nan})
    pd.DataFrame(crop_rows).to_csv(PAIR_CHECKPOINT,index=False)
    completed_pairs.add((a,b))
    PAIR_COMPLETED.write_text(json.dumps([list(x) for x in sorted(completed_pairs)],indent=2))
    print(f'{a}–{b}: {switched_ha:.1f} ha have a different mapped crop code.')

crop_switch_df = pd.DataFrame(crop_rows).sort_values('from_year')
crop_switch_df.to_csv(DIRS['Tables']/'consecutive_crop_switching_summary.csv',index=False)
if not crop_switch_df.empty:
    fig, ax = plt.subplots(figsize=(11,4.5))
    ax.bar(crop_switch_df.to_year.astype(int).astype(str),crop_switch_df.different_pct,
           color='#b77738',edgecolor='white')
    ax.set(xlabel='Second year in consecutive pair',
           ylabel='Area with different crop class (%)',
           title='Year-to-year mapped crop-class switching (crop-type classes in both years)')
    ax.tick_params(axis='x',rotation=45)
    save_figure('03_annual_crop_class_switching.png')
display(crop_switch_df.round(2))

## 9. Land-cover transition matrix: selected baseline and target years

We use a single encoded integer per pixel (`from_group × 100 + to_group`) to calculate transitions in one grouped area request. Each transition is measured in hectares. This matrix concerns **broad land-cover groups**; an accompanying class-level matrix is generated for the latest consecutive crop pair below.

In [ ]:
def encoded_pair(a,b,mode='broad',crop_only=False):
    if mode == 'broad':
        x,y = broad_aci(a),broad_aci(b)
        multiplier = 100
    elif mode == 'detailed':
        x,y = raw_aci(a),raw_aci(b)
        multiplier = 1000
    else:
        raise ValueError('mode must be broad or detailed.')
    encoded = x.multiply(multiplier).add(y).rename('transition_code').toInt32()
    valid = x.mask().And(y.mask())
    if crop_only:
        valid = valid.And(crop_flag(a)).And(crop_flag(b))
    return encoded.updateMask(valid)

TRANSITION_FILE = DIRS['Tables']/f'broad_transitions_{CHANGE_FROM}_{CHANGE_TO}.csv'
if TRANSITION_FILE.exists():
    transition_df = pd.read_csv(TRANSITION_FILE)
    print('Restored long-term transition CSV from current ROI checkpoint.')
else:
    rows = area_by_class(encoded_pair(CHANGE_FROM, CHANGE_TO))
    transition_df = pd.DataFrame([{'from_group':int(r['class_code'])//100,
                                    'to_group':int(r['class_code'])%100,
                                    'area_ha':r['area_ha']} for r in rows])
    transition_df.to_csv(TRANSITION_FILE,index=False)

if transition_df.empty:
    raise RuntimeError('No valid paired ACI pixels in your ROI.')
transition_df['from_name'] = transition_df.from_group.map(GROUP_NAMES)
transition_df['to_name'] = transition_df.to_group.map(GROUP_NAMES)
transition_df.to_csv(TRANSITION_FILE,index=False)
mat = transition_df.pivot_table(index='from_name',columns='to_name',
                                values='area_ha',aggfunc='sum',fill_value=0)
mat.to_csv(DIRS['Tables']/f'broad_transition_matrix_{CHANGE_FROM}_{CHANGE_TO}.csv')

# Display major categories while preserving every transition in the CSV.
major = (transition_df.groupby('from_group').area_ha.sum()
         .add(transition_df.groupby('to_group').area_ha.sum(),fill_value=0)
         .nlargest(9).index.astype(int).tolist())
selected = [GROUP_NAMES[i] for i in major]
heat = mat.reindex(index=selected,columns=selected,fill_value=0)
fig,ax=plt.subplots(figsize=(10.5,8))
im=ax.imshow(heat.values, cmap='YlGnBu',aspect='auto')
ax.set_xticks(range(len(selected)),labels=selected,rotation=50,ha='right')
ax.set_yticks(range(len(selected)),labels=selected)
ax.set(xlabel=f'Target year {CHANGE_TO}',ylabel=f'Baseline year {CHANGE_FROM}',
       title=f'Land-cover transition matrix ({CHANGE_FROM} → {CHANGE_TO}) — top groups')
fig.colorbar(im,ax=ax,label='Transition area (ha)',shrink=.78)
save_figure('04_landcover_transition_heatmap.png')
print('Saved full matrix and transition records:',TRANSITION_FILE)

## 10. Sankey diagrams: broad change and annual detailed crop changes

Sankey links display class-to-class area transfers across **two selected dates**, not the path followed in every intervening year. We additionally construct a **crop-code** Sankey for the most recent *consecutive* pair to demonstrate the difference between short-term class switching and long-term landscape dynamics. Low-area links may be filtered from interactive Sankeys for readability; complete transition CSVs retain all links.

In [ ]:
def make_sankey(frame, a, b, from_col, to_col, label_lookup,
                filename, title, min_ha=SANKey_MIN_FLOW_HA):
    if not HAS_PLOTLY or frame.empty:
        print('Sankey unavailable: Plotly missing or no transition records.')
        return
    df = frame[frame.area_ha >= min_ha].copy()
    if df.empty:
        print('No transitions pass the Sankey area threshold:',min_ha)
        return
    source_ids = sorted(df[from_col].unique())
    target_ids = sorted(df[to_col].unique())
    labels = [f'{a}: {label_lookup(i)}' for i in source_ids] +              [f'{b}: {label_lookup(i)}' for i in target_ids]
    source_idx={k:i for i,k in enumerate(source_ids)}
    target_idx={k:len(source_ids)+i for i,k in enumerate(target_ids)}
    fig=go.Figure(go.Sankey(
        node=dict(label=labels,pad=12,thickness=14),
        link=dict(source=[source_idx[x] for x in df[from_col]],
                  target=[target_idx[x] for x in df[to_col]],
                  value=df.area_ha.tolist(),
                  customdata=[f'{v:,.1f} ha' for v in df.area_ha],
                  hovertemplate='%{source.label} → %{target.label}<br>%{customdata}<extra></extra>')
    ))
    fig.update_layout(title=title, font_size=11, height=750)
    fig.write_html(str(DIRS['Figures']/filename),include_plotlyjs=True,full_html=True)
    display(fig)
    print('Saved interactive Sankey:',DIRS['Figures']/filename)

make_sankey(transition_df,CHANGE_FROM,CHANGE_TO,'from_group','to_group',
            lambda c: GROUP_NAMES.get(int(c),str(c)),
            '05_broad_landcover_sankey.html',
            f'Broad land-cover transitions, {CHANGE_FROM} → {CHANGE_TO}')

# Detailed transitions restricted to pixels classified as crop-type classes in both years.
if CONSECUTIVE_PAIRS:
    CROP_FROM,CROP_TO = CONSECUTIVE_PAIRS[-1]
    detail_path=DIRS['Tables']/f'detailed_crop_transitions_{CROP_FROM}_{CROP_TO}.csv'
    if detail_path.exists():
        detailed_trans_df=pd.read_csv(detail_path)
    else:
        detail_records=area_by_class(encoded_pair(CROP_FROM,CROP_TO,'detailed',crop_only=True))
        detailed_trans_df=pd.DataFrame([
            {'from_code':int(x['class_code'])//1000,
             'to_code':int(x['class_code'])%1000,
             'area_ha':x['area_ha']}
            for x in detail_records],columns=['from_code','to_code','area_ha'])
        detailed_trans_df.to_csv(detail_path,index=False)

    if not detailed_trans_df.empty:
        detailed_matrix=detailed_trans_df.pivot_table(index='from_code',columns='to_code',
                 values='area_ha',fill_value=0)
        detailed_matrix.to_csv(DIRS['Tables']/f'detailed_crop_transition_matrix_{CROP_FROM}_{CROP_TO}.csv')
        # Include 10 most represented crop codes; all other cropland codes become an 'Other crops' node.
        combined=(detailed_trans_df.groupby('from_code').area_ha.sum()
                  .add(detailed_trans_df.groupby('to_code').area_ha.sum(),fill_value=0))
        keep_codes=set(combined.nlargest(10).index.astype(int))
        summarized=detailed_trans_df.copy()
        summarized['from_plot']=summarized.from_code.apply(lambda x:int(x) if int(x) in keep_codes else 999)
        summarized['to_plot']=summarized.to_code.apply(lambda x:int(x) if int(x) in keep_codes else 999)
        summarized=summarized.groupby(['from_plot','to_plot'],as_index=False).area_ha.sum()
        make_sankey(summarized,CROP_FROM,CROP_TO,'from_plot','to_plot',
                    lambda c:'Other crop classes' if int(c)==999 else CLASS_INFO.get(int(c),(str(c),))[0],
                    '06_consecutive_detailed_crop_sankey.html',
                    f'Crop-class changes in consecutive years {CROP_FROM} → {CROP_TO} (top classes)')
        print('All detailed links retained in',detail_path)
else:
    print('No consecutive 2011+ pairs; skip detailed Sankey.')

## 11. Mapping change: endpoint difference versus multi-year persistence

We create three **separate** maps:

- **Endpoint change:** Broad-group class same/different at the two endpoint years (not definitive conversion).
- **Persistent transitions:** Early three *consecutive* annual inventories compared with the late three consecutive inventories. Candidate grassland→cropland and cropland→grassland transitions must be consistently classified in all three relevant years at both ends. These are **candidates** for field validation, not proof of historical conversion.
- **Consecutive crop-code switching frequency:** Number of year-to-year changes in the detailed cropland code divided by the number of valid adjacent-year pairs classified as crop-type classes in both years. Pixels without comparable pairs are masked.

A switch in crop type is **not** interpreted as grassland conversion.

In [ ]:
def all_true(images):
    return ee.ImageCollection.fromImages([ee.Image(im).unmask(0).toByte() for im in images]).min().eq(1)

# Endpoint change map: 1 = same broad class, 2 = different broad class.
first_group,last_group=broad_aci(CHANGE_FROM),broad_aci(CHANGE_TO)
endpoint_change=(ee.Image.constant(1).where(first_group.neq(last_group),2)
                 .updateMask(first_group.mask().And(last_group.mask()))
                 .rename('endpoint_change').toInt16())

EARLY_WINDOW = list(range(CHANGE_FROM,CHANGE_FROM+PERSISTENCE_WINDOW))
LATE_WINDOW = list(range(CHANGE_TO-PERSISTENCE_WINDOW+1,CHANGE_TO+1))
if not (all(y in YEARS for y in EARLY_WINDOW+LATE_WINDOW) and max(EARLY_WINDOW)<min(LATE_WINDOW)):
    print('Not enough distinct consecutive years for 3-year persistence. Using endpoint-only')
    print('persistent map will be masked and labelled unavailable.')
    PERSISTENCE_AVAILABLE=False
    persistent_map=None
else:
    PERSISTENCE_AVAILABLE=True
    all_window_years=EARLY_WINDOW+LATE_WINDOW
    # Invalid/unclassified pixels may not satisfy an all-years agreement.
    valid_all=all_true([raw_aci(y).mask() for y in all_window_years])
    early_grass=all_true([raw_aci(y).eq(GRASS_CODE) for y in EARLY_WINDOW])
    late_grass=all_true([raw_aci(y).eq(GRASS_CODE) for y in LATE_WINDOW])
    early_crop=all_true([cropland_landuse_flag(y) for y in EARLY_WINDOW])
    late_crop=all_true([cropland_landuse_flag(y) for y in LATE_WINDOW])
    persistent_map=(ee.Image.constant(5)
                    .where(early_grass.And(late_crop),1)
                    .where(early_crop.And(late_grass),2)
                    .where(early_grass.And(late_grass),3)
                    .where(early_crop.And(late_crop),4)
                    .updateMask(valid_all).rename('persistent_change').toInt16())
    print('Persistent-change comparison windows:',EARLY_WINDOW,'→',LATE_WINDOW)

PERSISTENT_NAMES={1:'Grassland → cropland candidate',2:'Cropland → grassland candidate',
                  3:'Consistent grassland',4:'Consistent cropland',
                  5:'Other / inconsistent pattern'}
PERSISTENT_COLORS=['#df7b3b','#56a1bd','#7bab45','#d3bc48','#b8b8b8']

valid_pair_masks=[]
switch_masks=[]
for a,b in CONSECUTIVE_PAIRS:
    valid=crop_flag(a).And(crop_flag(b)).unmask(0).toByte()
    changed=raw_aci(a).neq(raw_aci(b)).unmask(0).And(valid).toByte()
    valid_pair_masks.append(valid)
    switch_masks.append(changed)

if valid_pair_masks:
    N_CROP_PAIRS=ee.ImageCollection.fromImages(valid_pair_masks).sum().rename('valid_crop_pairs')
    N_SWITCHES=ee.ImageCollection.fromImages(switch_masks).sum().rename('switched_crop_pairs')
    SWITCH_FREQUENCY=(N_SWITCHES.divide(N_CROP_PAIRS)
                      .updateMask(N_CROP_PAIRS.gt(0))
                      .rename('crop_switch_frequency').toFloat())
else:
    SWITCH_FREQUENCY=None

if PERSISTENCE_AVAILABLE:
    persistent_stats=area_by_class(persistent_map)
    persistent_df=pd.DataFrame([{'change_code':r['class_code'],
            'change_label':PERSISTENT_NAMES.get(r['class_code'],'Other'),
            'area_ha':r['area_ha']} for r in persistent_stats])
    persistent_df.to_csv(DIRS['Tables']/'persistent_landcover_change_areas.csv',index=False)
    fig,ax=plt.subplots(figsize=(10,4.8))
    vals=[float(persistent_df.loc[persistent_df.change_code==i,'area_ha'].sum()) for i in range(1,6)]
    ax.barh([PERSISTENT_NAMES[i] for i in range(1,6)],vals,color=PERSISTENT_COLORS)
    ax.set(xlabel='Mapped area (ha)',
           title=f'Persistent land-cover patterns ({EARLY_WINDOW[0]}–{EARLY_WINDOW[-1]} vs {LATE_WINDOW[0]}–{LATE_WINDOW[-1]})')
    ax.invert_yaxis()
    save_figure('07_persistent_landcover_patterns_ha.png')
    display(persistent_df.round(2))
else:
    print('Persistent comparison not computed: insufficient contiguous years.')

## 12. Interactive inspection of ACI maps and transition products

Run this map cell to view your ROI, baseline and target grouped ACI classes and the change products. Click layers on/off in the layer control. Use the RGB imagery basemap only for geographic orientation; AAFC classes are the actual analytical data.

If map widgets or tile services are blocked in Colab, the **CSV statistics, figures and GeoTIFF exports still work**.

In [ ]:
try:
    m=geemap.Map(center=[49.60,-98.18],zoom=10,basemap='Esri.WorldImagery')
    m.centerObject(ROI_EE,11)
    broad_vis={'min':1,'max':len(GROUP_IDS),'palette':BROAD_PALETTE}
    m.addLayer(broad_aci(CHANGE_FROM).clip(ROI_EE),broad_vis,f'ACI broad {CHANGE_FROM}',True)
    m.addLayer(broad_aci(CHANGE_TO).clip(ROI_EE),broad_vis,f'ACI broad {CHANGE_TO}',False)
    m.addLayer(endpoint_change.clip(ROI_EE),
               {'min':1,'max':2,'palette':['#65ae79','#d26b6b']},
               'Same/different broad group at endpoints',False)
    if PERSISTENCE_AVAILABLE:
        m.addLayer(persistent_map.clip(ROI_EE),{'min':1,'max':5,'palette':PERSISTENT_COLORS},
                   'Persistent land-cover candidates',False)
    if SWITCH_FREQUENCY is not None:
        m.addLayer(SWITCH_FREQUENCY.clip(ROI_EE),
                   {'min':0,'max':1,'palette':['#ffffcc','#fd8d3c','#bd0026']},
                   'Crop class switching frequency',False)
    m.addLayer(ee.Image().byte().paint(ROI_EE,1,2),{'palette':['#ffffff']},'ROI outline',True)
    display(m)
except Exception as exc:
    print('Interactive map could not render:',exc)
    print('Proceed; tables, figures, and GeoTIFF exports do not depend on the widget.')

## 13. Optional static classified maps (PNG) saved in the student's Drive

For each requested map, Earth Engine renders a server-side visualization at **thumbnail resolution**. These PNGs are useful for the course report, but not for scientific raster calculations. A failed PNG thumbnail does not stop analysis. Use the GeoTIFF exports for GIS-quality pixels.

In [ ]:
STATIC_MAP_YEARS = sorted(set([CHANGE_FROM, (CHANGE_FROM+CHANGE_TO)//2, CHANGE_TO]) & set(YEARS))

def save_ee_quicklook(image, title, basename, palette, vmin, vmax, legend_labels=None):
    vis=ee.Image(image).visualize(min=vmin,max=vmax,palette=palette)
    thumb_url=vis.getThumbURL({'region':ROI_EE.bounds(maxError=1),
                               'dimensions':1000,'format':'png'})
    req=urllib.request.Request(thumb_url,headers={'User-Agent':'Mozilla/5.0'})
    with urllib.request.urlopen(req,timeout=100) as response:
        pic=Image.open(BytesIO(response.read())).convert('RGBA')
    fig,ax=plt.subplots(figsize=(10,8))
    ax.imshow(pic,interpolation='nearest')
    ax.set_title(title)
    ax.axis('off')
    if legend_labels:
        handles=[Patch(facecolor=palette[i],edgecolor='gray',label=label)
                 for i,label in enumerate(legend_labels)]
        ax.legend(handles=handles,loc='center left',bbox_to_anchor=(1.0,.5),fontsize=8)
    fig.savefig(DIRS['Maps']/basename,dpi=300,bbox_inches='tight')
    plt.close(fig)

if MAKE_STATIC_MAPS:
    for year in STATIC_MAP_YEARS:
        try:
            save_ee_quicklook(broad_aci(year).clip(ROI_EE),
                f'AAFC crop inventory — broad classes, {year}',f'broad_ACI_{year}.png',
                BROAD_PALETTE,1,len(GROUP_IDS),[GROUP_NAMES[i] for i in GROUP_IDS])
            print('Saved broad ACI quicklook:',year)
        except Exception as exc:
            print(f'Quicklook {year} unavailable (continuing): {str(exc)[:180]}')
    if PERSISTENCE_AVAILABLE:
        try:
            save_ee_quicklook(persistent_map.clip(ROI_EE),
                 'Persistent land-cover change candidates','persistent_change_map.png',
                 PERSISTENT_COLORS,1,5,[PERSISTENT_NAMES[i] for i in range(1,6)])
            print('Saved persistence map PNG.')
        except Exception as exc:
            print('Persistence quicklook unavailable:',str(exc)[:180])
    if SWITCH_FREQUENCY is not None:
        try:
            save_ee_quicklook(SWITCH_FREQUENCY.clip(ROI_EE),
                 'Relative frequency of changing detailed cropland class',
                 'crop_switch_frequency_map.png',
                 ['#ffffcc','#fd8d3c','#bd0026'],0,1)
        except Exception as exc:
            print('Switch quicklook unavailable:',str(exc)[:180])
else:
    print('Quicklooks disabled. Interactive maps and scientific GeoTIFFs remain available.')

## 14. Scientific GeoTIFF exports to each student's Google Drive

Earth Engine exports are **asynchronous tasks**. They start in a **separate top-level Google Drive folder** (not inside mounted Drive paths). By default we export baseline and target detailed ACI maps, endpoint change, candidate persistent change, and crop-switching frequency. Optionally enable all annual detailed raster exports. No resampling is interpreted as new information; categorical images use default nearest-neighbour export behaviour.

**Note:** The 2010 original imagery is about 56 m even if an export is requested on a 30 m grid. Never treat those pixels as an independent 30 m measurement.

In [ ]:
EXPORT_DRIVE_FOLDER = f'P03_{TAG}_{roi_fingerprint}_GeoTIFF'
EXPORT_TASKS = []

def start_geotiff(image, short_name, scale=PROCESSING_SCALE_M):
    task_name = f'P03_{TAG}_{roi_fingerprint}_{short_name}'[:100]
    task=ee.batch.Export.image.toDrive(
        image=ee.Image(image).clip(ROI_EE),description=task_name,
        folder=EXPORT_DRIVE_FOLDER,fileNamePrefix=task_name,
        region=ROI_EE,scale=scale,maxPixels=1e10,fileFormat='GeoTIFF')
    task.start()
    EXPORT_TASKS.append({'name':task_name,'task':task})
    print('Submitted:',task_name)

if START_GEOTIFF_EXPORTS:
    years_to_export = YEARS if EXPORT_ALL_ANNUAL_GEOTIFFS else sorted({CHANGE_FROM,CHANGE_TO})
    for year in years_to_export:
        start_geotiff(raw_aci(year).rename('ACI_landcover').toInt16(),f'ACI_detailed_{year}')
    start_geotiff(endpoint_change.toInt16(),f'endpoint_broad_changed_{CHANGE_FROM}_{CHANGE_TO}')
    if PERSISTENCE_AVAILABLE:
        start_geotiff(persistent_map.toInt16(),f'persistent_change_{CHANGE_FROM}_{CHANGE_TO}')
    if SWITCH_FREQUENCY is not None:
        start_geotiff(SWITCH_FREQUENCY.toFloat(),f'crop_switch_frequency_{min(YEARS)}_{max(YEARS)}')
else:
    print('Earth Engine GeoTIFF task creation disabled.')

print('Expected completed GeoTIFFs in My Drive /',EXPORT_DRIVE_FOLDER)
print('Tasks submitted this session:',len(EXPORT_TASKS))

In [ ]:
# You may rerun this check cell later; it does not submit new tasks.
export_status = []
for item in EXPORT_TASKS:
    try:
        status = item['task'].status()
        export_status.append({'task_name':item['name'],'state':status.get('state'),
            'id':status.get('id'),'error':status.get('error_message','')})
    except Exception as exc:
        export_status.append({'task_name':item['name'],'state':'UNKNOWN','error':str(exc)})
if export_status:
    export_df=pd.DataFrame(export_status)
    export_df.to_csv(DIRS['GeoTIFF_Logs']/'export_task_status.csv',index=False)
    display(export_df)
else:
    print('No tasks created in this session.')

## 15. Export manifest, reproducibility metadata, and deliverables

Save the exact ROI, available years, missing EE years, thematic grouping crosswalk, scale, change years, and output manifest. CSV/PNG/HTML/GeoJSON data already reside in your own Google Drive; the GeoTIFF tasks can continue after the notebook cell finishes, subject to EE/Drive task rules. **Do not submit an unfinished GeoTIFF task as a completed file.**

In [ ]:
settings = {
    'student_id':STUDENT_ID,'ee_project':EE_PROJECT_ID,
    'aci_collection':'AAFC/ACI','band':'landcover',
    'requested_start_year':START_YEAR,'requested_end_year':END_YEAR,
    'available_years_used':YEARS,'unavailable_years':MISSING_YEARS,
    'change_from':CHANGE_FROM,'change_to':CHANGE_TO,
    'native_resolution_2010_m_approx':56,
    'native_resolution_2011_onward_m_approx':30,
    'processing_scale_m':PROCESSING_SCALE_M,
    'early_persistence_years':EARLY_WINDOW,'late_persistence_years':LATE_WINDOW,
    'persistence_available':PERSISTENCE_AVAILABLE,
    'year_to_year_crop_switching_from_2011':True,
    'valid_area_percentage_denominator':'valid classified pixels (excluding cloud code 10)',
    'roi_area_km2':ROI_AREA_KM2,'roi_fingerprint':roi_fingerprint,
    'roi_geojson_path':str(DIRS['ROI']/'study_area.geojson'),
    'drive_output_path':str(RUN_ROOT),
    'geotiff_drive_folder':EXPORT_DRIVE_FOLDER,
    'export_all_annual_geotiffs':EXPORT_ALL_ANNUAL_GEOTIFFS,
    'processing_utc':datetime.now(timezone.utc).isoformat(),
    'reference':'https://developers.google.com/earth-engine/datasets/catalog/AAFC_ACI'
}
(RUN_ROOT/'run_configuration.json').write_text(json.dumps(settings,indent=2))
(DIRS['Tables']/'teaching_crosswalk.json').write_text(json.dumps({
    'AAFC_class_info':{str(k):{'name':v[0],'color':v[1]} for k,v in CLASS_INFO.items()},
    'broad_groups': [{'id':gid,'name':name,'color':color,'AAFC_codes':codes}
                     for gid,name,color,codes in GROUP_DEF],
    'crop_group_ids_for_type_switching':CROP_GROUP_IDS,
    'crop_type_codes':CROP_CODES,
    'cropland_landuse_codes_for_persistence':CROPLAND_LANDUSE_CODES,
    'caution':'Broad groups are course-defined, not official AAFC classes'
},indent=2))
manifest=[{'path':str(p.relative_to(RUN_ROOT)),'size_bytes':p.stat().st_size}
          for p in RUN_ROOT.rglob('*') if p.is_file()]
pd.DataFrame(manifest).to_csv(RUN_ROOT/'output_manifest.csv',index=False)
print('All immediate data and graphics stored in your personal Google Drive:')
print(RUN_ROOT)
print('GeoTIFF tasks (if completed): My Drive /',EXPORT_DRIVE_FOLDER)
print('Files in current manifest:',len(manifest))

## 16. Student interpretation and submission

**Submit** a single PDF report (approximately 700–1000 words, excluding figure captions and references) that includes:

1. **Study area:** A short justification for choosing your ROI and the selected year range. Report the polygon area and any missing ACI years.
2. **Annual land-cover dynamics:** One annual percentage chart, one chart of the most important detailed crops/classes, and a table giving class areas (ha) for at least three years.
3. **Crop switching:** A consecutive-year crop-switching chart and a discussion of why a mapped crop-code change is **not** proof of crop rotation or management decisions.
4. **Change detection:** A complete broad-group transition matrix for the two chosen endpoints and an interactive Sankey exported as HTML (or your transition CSV if Plotly is unavailable).
5. **Persistent conversion:** A map and small table of the multi-year persistent categories (when enough contiguous ACI years exist), explaining why candidates require validation.
6. **Uncertainty:** Discuss the 2010 native-resolution mismatch, nominal categorical classes, classification uncertainty, ROI size, missed years, and possible misinterpretation of grassland/pasture/cropland.

**Submission file:** `P03_FirstName_LastName.pdf` (submit through the course assignment area, not by sharing personal Drive permissions). Retain the Colab notebook and all reproducible outputs in **your own Google Drive**.

### Scientific discussion prompts

- Is a switch from one crop class to another equivalent to grassland conversion? Why not?
- How would three-year persistence reduce, but not remove, false change detections?
- How might the 2010 inventory resolution affect an analysis spanning 2010–2025?
- Which broad classes have the largest gains/losses, and what alternative explanations should be considered?
- What additional field, crop insurance, or independent satellite data would you need to strengthen conclusions?

### Sources
- Google Earth Engine Data Catalog — **Canada AAFC Annual Crop Inventory**: https://developers.google.com/earth-engine/datasets/catalog/AAFC_ACI
- Agriculture and Agri-Food Canada — **Annual Crop Inventory 2025**: https://open.canada.ca/data/en/dataset/89073037-d2fc-4a6d-9ce9-f0b1bcb48745
- Earth Engine guide — **Statistics of an image region**: https://developers.google.com/earth-engine/guides/reducers_reduce_region

**AI use:** AI may help develop or debug code, but each student is responsible for understanding the methodology, independently interpreting results, and justifying their scientific conclusions.